# A guide Portfolio Optimization Environment

This notebook aims to provide an example of using PortfolioOptimizationEnv (or POE) to train a reinforcement learning model that learns to solve the portfolio optimization problem.

In this document, we will reproduce a famous architecture called EIIE (ensemble of identical independent evaluators), introduced in the following paper:

- Zhengyao Jiang, Dixing Xu, & Jinjun Liang. (2017). A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem. https://doi.org/10.48550/arXiv.1706.10059.

It's advisable to read it to understand the algorithm implemented in this notebook.

### Note
If you're using this environment, consider citing the following paper (in adittion to FinRL references):

- Caio Costa, & Anna Costa (2023). POE: A General Portfolio Optimization Environment for FinRL. In *Anais do II Brazilian Workshop on Artificial Intelligence in Finance* (pp. 132–143). SBC. https://doi.org/10.5753/bwaif.2023.231144.

```
@inproceedings{bwaif,
 author = {Caio Costa and Anna Costa},
 title = {POE: A General Portfolio Optimization Environment for FinRL},
 booktitle = {Anais do II Brazilian Workshop on Artificial Intelligence in Finance},
 location = {João Pessoa/PB},
 year = {2023},
 keywords = {},
 issn = {0000-0000},
 pages = {132--143},
 publisher = {SBC},
 address = {Porto Alegre, RS, Brasil},
 doi = {10.5753/bwaif.2023.231144},
 url = {https://sol.sbc.org.br/index.php/bwaif/article/view/24959}
}

```

## Installation and imports

To run this notebook in google colab, uncomment the cells below.

In [ ]:
## install finrl library
# !sudo apt install swig
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

In [ ]:
## We also need to install quantstats, because the environment uses it to plot graphs
# !pip install quantstats

In [ ]:
## Hide matplotlib warnings
# import warnings
# warnings.filterwarnings('ignore')

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

#### Import the necessary code libraries

In [ ]:
%pip install webdriver_manager

In [ ]:
%pip install quantstats --upgrade --no-cache-dir


In [ ]:
%pip install torch_geometric

In [ ]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -U IPython matplotlib ipykernel matplotlib-inline quantstats

In [ ]:
import torch

from sklearn.preprocessing import MaxAbsScaler

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import GroupByScaler
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl.agents.portfolio_optimization.models import DRLAgent
from FinRL.finrl.agents.portfolio_optimization.architectures_changed import EIIE

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [ ]:
from FinRL.finrl.agents.portfolio_optimization.architectures_changed import EI3

In [ ]:
!python.exe -m pip install --upgrade pip

## Fetch data

In his paper, *Jiang et al* creates a portfolio composed by the top-11 cryptocurrencies based on 30-days volume. Since it's not specified when this classification was done, it's difficult to reproduce, so we will use a similar approach in the Brazillian stock market:

- We select top-10 stocks from Brazillian stock market;
- For simplicity, we disconsider stocks that have missing data for the days in period 2011-01-01 to 2019-12-31 (9 years);

In [ ]:
TOP_BRL = [
    "VALE3.SA", "PETR4.SA", "ITUB4.SA", "BBDC4.SA",
    "BBAS3.SA", "RENT3.SA", "LREN3.SA", "PRIO3.SA",
    "WEGE3.SA", "ABEV3.SA"
]

In [ ]:
TOP_BRL = [
        "601012.SS",
        "600887.SS",
        "600030.SS",   
        "600031.SS",
        "600036.SS",        
        "600276.SS", 
        "600519.SS", 
        "601318.SS", 
        "600703.SS", 
        "601166.SS", 
    ]


In [ ]:
%env HTTP_PROXY=http://127.0.0.1:7897
%env HTTPS_PROXY=http://127.0.0.1:7897

In [ ]:
print(len(TOP_BRL))

portfolio_raw_df = YahooDownloader(start_date = '2014-01-01',
                                end_date = '2024-12-31',
                                ticker_list = TOP_BRL).fetch_data()
portfolio_raw_df

In [ ]:
from finrl.meta.preprocessor.preprocessors import FeatureEngineer

fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=["macd", "rsi_30", "cci_30", "dx_30"],
    use_turbulence=False,
    user_defined_feature=False
)

portfolio_raw_df = fe.preprocess_data(portfolio_raw_df.copy())
portfolio_raw_df

In [ ]:
portfolio_raw_df.groupby("tic").count()

### Normalize Data

We normalize the data dividing the time series of each stock by its maximum value, so that the dataframe contains values between 0 and 1.

In [ ]:
import numpy as np
import pandas as pd

def safe_log_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """
    Return log(numerator / denominator) only when both > 0, otherwise NaN.
    """
    out = pd.Series(np.nan, index=numerator.index, dtype="float64")
    mask = (numerator > 0) & (denominator > 0)
    out.loc[mask] = np.log(numerator.loc[mask] / denominator.loc[mask])
    return out

def add_custom_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["tic", "date"]).reset_index(drop=True)

    g = df.groupby("tic", group_keys=False)

    # ---- lagged close for returns ----
    close_lag1 = g["close"].shift(1)
    close_lag5 = g["close"].shift(5)
    close_lag10 = g["close"].shift(10)
    close_lag20 = g["close"].shift(20)

    # ---- log returns ----
    df["r1"] = safe_log_ratio(df["close"], close_lag1)
    df["r5"] = safe_log_ratio(df["close"], close_lag5)
    df["r10"] = safe_log_ratio(df["close"], close_lag10)
    df["r20"] = safe_log_ratio(df["close"], close_lag20)

    # ---- intraday / overnight / range ----
    df["intraday"] = safe_log_ratio(df["close"], df["open"])
    df["overnight"] = safe_log_ratio(df["open"], close_lag1)
    df["range"] = safe_log_ratio(df["high"], df["low"])

    return df

In [ ]:
df_portfolio = portfolio_raw_df[["date", "tic", "open", "close", "high", "low", "volume", "macd", "rsi_30", "cci_30", "dx_30"]]
df_portfolio = add_custom_features(df_portfolio)

df_portfolio["cci_30"] = df_portfolio["cci_30"].clip(-300, 300)
feature_cols = ["close", "macd", "rsi_30", "cci_30", "dx_30",
                "r1", "r5", "r10", "r20", 
                "intraday", "overnight", "range"]

for col in feature_cols:
    df_portfolio[col] = GroupByScaler(
        by="tic", scaler=MaxAbsScaler
    ).fit_transform(df_portfolio[["tic", col]])[col]
df_portfolio

In [ ]:
df_portfolio_train = df_portfolio[(df_portfolio["date"] >= "2014-01-30") & (df_portfolio["date"] < "2021-12-31")]
df_portfolio_2022 = df_portfolio[(df_portfolio["date"] >= "2022-01-01") & (df_portfolio["date"] < "2022-12-31")]
df_portfolio_2023 = df_portfolio[(df_portfolio["date"] >= "2023-01-01") & (df_portfolio["date"] < "2023-12-31")]
df_portfolio_2024 = df_portfolio[(df_portfolio["date"] >= "2024-01-01") & (df_portfolio["date"] < "2024-12-31")]

In [ ]:
df_portfolio_train

In [ ]:
print(df_portfolio_train.isnull().sum())
print((df_portfolio_train == float('inf')).sum())
print((df_portfolio_train == float('-inf')).sum())
print(df_portfolio_train.describe())

In [ ]:
# nan_rows = df_portfolio_train[df_portfolio_train["vol_ratio20"].isna()][["date", "tic", "vol_ratio20"]]
# print(nan_rows)
# print(f"共 {len(nan_rows)} 行")

### Instantiate Environment

Using the `PortfolioOptimizationEnv`, it's easy to instantiate a portfolio optimization environment for reinforcement learning agents. In the example below, we use the dataframe created before to start an environment.

In [ ]:
environment = PortfolioOptimizationEnv(
        df_portfolio_train,
        initial_amount=100000,
        comission_fee_pct=0.002,
        time_window=50,
        features=[
    "macd",
    "rsi_30",
    "cci_30",
    "dx_30",
    "close",
    "r1",
    "r5",
    "r10",
    "r20",
    "intraday",
    "overnight",
    "range",
],
        normalize_df=None
    )

### Instantiate Model

Now, we can instantiate the model using FinRL API. In this example, we are going to use the EIIE architecture introduced by Jiang et. al.

:exclamation: **Note:** Remember to set the architecture's `time_window` parameter with the same value of the environment's `time_window`.

In [ ]:
# set PolicyGradient parameters
model_kwargs = {
    "lr": 1e-3,
    "policy": EI3,
}

# here, we can set EI3's parameters
policy_kwargs = {
    "initial_features": 12,   # 很重要：等于你的输入特征数
    "k_short": 3,
    "k_medium": 21,
    "conv_mid_features": 3,
    "conv_final_features": 20,
    "time_window": 50,
}

model = DRLAgent(environment).get_model("pg", device, model_kwargs, policy_kwargs)

### Train Model

In [ ]:
DRLAgent.train_model(model, episodes=5)

### Save Model

In [ ]:
torch.save(model.train_policy.state_dict(), "policy_EI3.pt")

## Test Model

### Instantiate different environments

Since we have three different periods of time, we need three different environments instantiated to simulate them.

In [ ]:
env_ei3_2022 = PortfolioOptimizationEnv(
    df_portfolio_2022,
    initial_amount=100000,
    comission_fee_pct=0.002,
    time_window=50,
    features=[
    "close",
    "macd",
    "rsi_30",
    "cci_30",
    "dx_30",
    "r1",
    "r5",
    "r10",
    "r20",
    "intraday",
    "overnight",
    "range",
],
    normalize_df=None
)

env_ei3_2023 = PortfolioOptimizationEnv(
    df_portfolio_2023,
    initial_amount=100000,
    comission_fee_pct=0.002,
    time_window=50,
    features=[
    "close",
    "macd",
    "rsi_30",
    "cci_30",
    "dx_30",
    "r1",
    "r5",
    "r10",
    "r20",
    "intraday",
    "overnight",
    "range",
],  
    normalize_df=None
)

env_ei3_2024 = PortfolioOptimizationEnv(
    df_portfolio_2024,
    initial_amount=100000,
    comission_fee_pct=0.002,
    time_window=50,
    features=[
    "close",
    "macd",
    "rsi_30",
    "cci_30",
    "dx_30",
    "r1",
    "r5",
    "r10",
    "r20",
    "intraday",
    "overnight",
    "range",
],
    normalize_df=None
)

### Test EIIE/EI3 architecture
Now, we can test the EIIE architecture in the three different test periods. It's important no note that, in this code, we load the saved policy even though it's not necessary just to show how to save and load your model.

In [ ]:
EI3_results = {
    "training": environment._asset_memory["final"],
    "2022": {},
    "2023": {},
    "2024": {}
}

# instantiate an architecture with the same arguments used in training
# and load with load_state_dict.
policy = EI3(
    initial_features=12,
    k_short=3,
    k_medium=21,
    conv_mid_features=3,
    conv_final_features=20,
    time_window=50,
    device=device,
)
policy.load_state_dict(torch.load("policy_EI3.pt"))

# 2022
DRLAgent.DRL_validation(model, env_ei3_2022, policy=policy)
EI3_results["2022"] = {
    "value": list(env_ei3_2022._asset_memory["final"]),
    "actions": np.array(env_ei3_2022._actions_memory).copy(),
}
# 2023
DRLAgent.DRL_validation(model, env_ei3_2023, policy=policy)
EI3_results["2023"] = {
    "value": list(env_ei3_2023._asset_memory["final"]),
    "actions": np.array(env_ei3_2023._actions_memory).copy(),
}

# 2024
DRLAgent.DRL_validation(model, env_ei3_2024, policy=policy)
EI3_results["2024"] = {
    "value": list(env_ei3_2024._asset_memory["final"]),
    "actions": np.array(env_ei3_2024._actions_memory).copy(),
}

### Test Uniform Equal Weight
For comparison, we will also test the performance of a uniform Equal Weight strategy. In this strategy, the portfolio has no remaining cash and the same percentage of money is allocated in each asset.

In [ ]:
EW_results = {
    "train": {},
    "2022": {},
    "2023": {},
    "2024": {}
}

PORTFOLIO_SIZE = len(TOP_BRL)

# train period
terminated = False
environment.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment.step(action)
EW_results["train"]["value"] = environment._asset_memory["final"]

# 2022
terminated = False
env_ei3_2022.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = env_ei3_2022.step(action)
EW_results["2022"]["value"] = env_ei3_2022._asset_memory["final"]

# 2023
terminated = False
env_ei3_2023.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = env_ei3_2023.step(action)
EW_results["2023"]["value"] = env_ei3_2023._asset_memory["final"]

# 2024
terminated = False
env_ei3_2024.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = env_ei3_2024.step(action)
EW_results["2024"]["value"] = env_ei3_2024._asset_memory["final"]

### Plot graphics

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 

plt.plot(EW_results["train"]["value"], label="Equal Weight")
plt.plot(EI3_results["training"], label="EI3")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in training period")
plt.legend()

plt.show()

In [ ]:
plt.plot(EW_results["2022"]["value"], label="Equal Weight")
plt.plot(EI3_results["2022"]["value"], label="EI3")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2022")
plt.legend()

plt.show()

plt.plot(EW_results["2023"]["value"], label="Equal Weight")
plt.plot(EI3_results["2023"]["value"], label="EI3")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2023")
plt.legend()

plt.show()

plt.plot(EW_results["2024"]["value"], label="Equal Weight")
plt.plot(EI3_results["2024"]["value"], label="EI3")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2024")
plt.legend()

plt.show()

We can see that the agent is able to learn a good policy but its performance is worse the more the test period advances into the future. To get a better performance in 2022, for example, the agent should probably be trained again using more recent data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_actions(actions, raw_df, title, save_path=None):
    actions = np.asarray(actions, dtype=np.float32)

    dates = sorted(pd.to_datetime(raw_df["date"]).unique())
    n_steps = actions.shape[0]
    x_dates = dates[-n_steps:] if len(dates) >= n_steps else np.arange(n_steps)

    tics = sorted(raw_df["tic"].unique().tolist())
    labels = ["cash"] + tics if actions.shape[1] == len(tics) + 1 else [f"asset_{i}" for i in range(actions.shape[1])]

    plt.figure(figsize=(14, 8))
    for i in range(actions.shape[1]):
        plt.plot(x_dates, actions[:, i], linewidth=1.2, label=labels[i])

    plt.xlabel("Date")
    plt.ylabel("Weight")
    plt.title(title)
    plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_actions(EI3_results["2022"]["actions"], df_portfolio_2022, "EI3 Weights 2022")
plot_actions(EI3_results["2023"]["actions"], df_portfolio_2023, "EI3 Weights 2023")
plot_actions(EI3_results["2024"]["actions"], df_portfolio_2024, "EI3 Weights 2024")
